# DailyDialog Dataset — Exploration & Data Pipeline

This notebook loads the DailyDialog dataset, explores its structure, builds a vocabulary, extracts conversation pairs, and creates PyTorch DataLoaders for the reply generator.

In [ ]:
import pandas as pd
import json
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import sys
sys.path.append('..')
import config

## 1. Load Data

In [ ]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("validation.csv")
test_df  = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)
train_df.head()

## 2. Parse Dialog Turns and Labels

In [ ]:
def clean_turns(dialog):
    """Split a dialog string into individual turns."""
    dialog = dialog.replace("\n", " ")
    turns = dialog.split("  ")
    return [t.strip() for t in turns if t.strip()]

def parse_labels(label_str):
    """Parse label string like '[0 4 2]' into list of ints."""
    return [int(x) for x in label_str.strip("[]").split()]

for df in [train_df, val_df, test_df]:
    df["turns"]          = df["dialog"].apply(clean_turns)
    df["act_labels"]     = df["act"].apply(parse_labels)
    df["emotion_labels"] = df["emotion"].apply(parse_labels)

print("Sample turns from first conversation:")
for i, turn in enumerate(train_df["turns"].iloc[0][:4]):
    print(f"  Turn {i+1}: {turn}")

## 3. Dataset Statistics

In [ ]:
print("=== Dataset Statistics ===")
print(f"Train conversations : {len(train_df)}")
print(f"Val conversations   : {len(val_df)}")
print(f"Test conversations  : {len(test_df)}")

avg_turns = train_df["turns"].apply(len).mean()
print(f"\nAvg turns per conversation : {avg_turns:.2f}")

all_turns = [t for conv in train_df["turns"] for t in conv]
avg_words = sum(len(t.split()) for t in all_turns) / len(all_turns)
print(f"Avg words per turn         : {avg_words:.2f}")
print(f"Total utterances in train  : {len(all_turns)}")

## 4. Emotion Distribution — Mapped to TextMe 18 Mood Classes

In [ ]:
DAILYDIALOG_EMOTION_MAP = {
    0: "casual",
    1: "angry",
    2: "angry",
    3: "anxious",
    4: "excited",
    5: "emotional",
    6: "excited"
}

all_emotions = [e for conv in train_df["emotion_labels"] for e in conv]
emotion_counts = Counter(all_emotions)

print("=== Emotion Label Distribution (Train) ===")
for label in sorted(emotion_counts.keys()):
    mood  = DAILYDIALOG_EMOTION_MAP[label]
    count = emotion_counts[label]
    print(f"  Label {label} -> {mood:12s} : {count:6d} samples")

labels = [f"{k}:{DAILYDIALOG_EMOTION_MAP[k]}" for k in sorted(emotion_counts.keys())]
counts = [emotion_counts[k] for k in sorted(emotion_counts.keys())]

plt.figure(figsize=(10, 5))
plt.bar(labels, counts, color='steelblue')
plt.title("DailyDialog Emotion Distribution")
plt.xlabel("Label : Mood Class")
plt.ylabel("Count")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
print("Note: Label 0 (neutral/casual) dominates — class imbalance expected.")

## 5. Extract Input/Response Pairs

In [ ]:
def clean_text(t):
    """Clean a single dialog turn string."""
    t = t.strip()
    t = t.lstrip("['").rstrip("']")
    t = t.strip("'").strip('"')
    t = t.replace(" ,", ",").replace(" .", ".")
    t = t.replace(" ?", "?").replace(" !", "!")
    t = " ".join(t.split())
    return t

def extract_pairs(df):
    """Extract consecutive (input, response) pairs from all conversations."""
    pairs = []
    for conv in df["turns"]:
        if len(conv) < 2:
            continue
        for i in range(len(conv) - 1):
            pairs.append({
                "input":    clean_text(conv[i]),
                "response": clean_text(conv[i+1])
            })
    return pairs

train_pairs = extract_pairs(train_df)
val_pairs   = extract_pairs(val_df)
test_pairs  = extract_pairs(test_df)

print(f"Train pairs : {len(train_pairs)}")
print(f"Val pairs   : {len(val_pairs)}")
print(f"Test pairs  : {len(test_pairs)}")
print(f"\nSample pair:")
print(f"  Input   : {train_pairs[0]['input']}")
print(f"  Response: {train_pairs[0]['response']}")

## 6. Build Shared Vocabulary

In [ ]:
PAD_TOKEN = config.PAD_TOKEN
SOS_TOKEN = config.SOS_TOKEN
EOS_TOKEN = config.EOS_TOKEN
UNK_TOKEN = config.UNK_TOKEN

def build_vocab(pairs, min_freq=2):
    """
    Build vocabulary from training pairs only.
    Words appearing less than min_freq times are excluded.
    """
    counter = Counter()
    for pair in pairs:
        counter.update(word_tokenize(pair["input"].lower()))
        counter.update(word_tokenize(pair["response"].lower()))

    vocab = {
        PAD_TOKEN: 0,
        SOS_TOKEN: 1,
        EOS_TOKEN: 2,
        UNK_TOKEN: 3
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(train_pairs, min_freq=2)

print(f"Vocabulary size : {len(vocab)}")
print(f"Special tokens  : PAD=0, SOS=1, EOS=2, UNK=3")
print(f"Sample entries  : {list(vocab.items())[4:12]}")

## 7. Tokenize and Pad

In [ ]:
MAX_LEN = config.MAX_SEQ_LEN

def tokenize_and_pad(text, vocab, max_len):
    """
    Tokenize text, convert to indices, add SOS/EOS tokens, and pad to max_len.
    """
    tokens  = word_tokenize(text.lower())
    indices = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens]
    indices = indices[:max_len - 2]
    indices = [vocab[SOS_TOKEN]] + indices + [vocab[EOS_TOKEN]]
    padding = [vocab[PAD_TOKEN]] * (max_len - len(indices))
    return indices + padding

sample = tokenize_and_pad(train_pairs[0]["input"], vocab, MAX_LEN)
print(f"Input text    : {train_pairs[0]['input']}")
print(f"Padded length : {len(sample)}")
print(f"First 10 idx  : {sample[:10]}")

## 8. PyTorch Dataset and DataLoader

In [ ]:
class DailyDialogDataset(Dataset):
    """
    PyTorch Dataset for DailyDialog conversation pairs.
    Each item returns (input_tensor, response_tensor).
    Both tensors have shape [MAX_SEQ_LEN].
    """
    def __init__(self, pairs, vocab, max_len):
        self.pairs   = pairs
        self.vocab   = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        src  = tokenize_and_pad(pair["input"],    self.vocab, self.max_len)
        tgt  = tokenize_and_pad(pair["response"], self.vocab, self.max_len)
        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long)
        )

train_dataset = DailyDialogDataset(train_pairs, vocab, MAX_LEN)
val_dataset   = DailyDialogDataset(val_pairs,   vocab, MAX_LEN)
test_dataset  = DailyDialogDataset(test_pairs,  vocab, MAX_LEN)

BATCH_SIZE = config.BATCH_SIZE

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

src_batch, tgt_batch = next(iter(train_loader))
print(f"Input batch shape   : {src_batch.shape}")
print(f"Response batch shape: {tgt_batch.shape}")
print("DataLoaders created successfully")

## 9. Verify contacts.json

In [ ]:
# Read contacts.json from repo root
# Do NOT recreate or overwrite it here

with open("../contacts.json", "r") as f:
    contacts = json.load(f)

print(f"Total contacts defined: {len(contacts['contacts'])}")
print("\nAll contact types:")
for c in contacts["contacts"]:
    print(f"  {c['name']:15s} -> {c['relation']}")

## 10. Summary

In [ ]:
print("=== DailyDialog Pipeline Summary ===")
print(f"Train conversations : {len(train_df)}")
print(f"Val conversations   : {len(val_df)}")
print(f"Test conversations  : {len(test_df)}")
print(f"Train pairs         : {len(train_pairs)}")
print(f"Val pairs           : {len(val_pairs)}")
print(f"Test pairs          : {len(test_pairs)}")
print(f"Vocabulary size     : {len(vocab)}")
print(f"Max sequence length : {MAX_LEN}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Input batch shape   : {list(src_batch.shape)}")
print(f"Response batch shape: {list(tgt_batch.shape)}")
print("\nEmotion label mapping (DailyDialog -> TextMe):")
for k, v in DAILYDIALOG_EMOTION_MAP.items():
    print(f"  {k} -> {v}")